# YerelAI Studio — Gemma 4 Google Colab LoRA Eğitimi

Bu defter, **YerelAI Studio** arayüzünden dışa aktardığınız `yerelai_unsloth_dataset.jsonl` dosyasını kullanarak ücretsiz **Google Colab T4 GPU** üzerinde **Gemma 4 (E2B veya E4B)** modelini **Unsloth QLoRA** ile hızlı ve düşük VRAM kullanarak eğitir.

Eğitim sonunda çıkan `yerelai_lora_adapter.zip` dosyasını indirip bilgisayarınızdaki `yerelai/models/loras/<adaptor_adi>/` klasörüne çıkartmanız yeterlidir; YerelAI Studio bu LoRA adaptörünü otomatik algılar.

In [ ]:
# 1. Unsloth ve Gerekli Kütüphanelerin Kurulumu (Colab T4)
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
# 2. Model Seçimi ve Yükleme (E2B: 4GB/8GB uyumlu, E4B: 8GB Turbo)
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"  # veya "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 3. YerelAI Studio'dan İndirilen JSONL Eğitim Setini Yükle
from google.colab import files
from datasets import load_dataset

uploaded = files.upload()  # yerelai_unsloth_dataset.jsonl dosyasını seçin
dataset_path = list(uploaded.keys())[0]
raw_ds = load_dataset("json", data_files=dataset_path, split="train")

def format_row(example):
    msgs = example["messages"]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = raw_ds.map(format_row)
print(f"Toplam eğitim örneği: {len(train_ds)}")

In [ ]:
# 4. QLoRA Eğitimi Başlat
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print("Eğitim tamamlandı!", trainer_stats)

In [ ]:
# 5. LoRA Adaptörünü Kaydet ve ZIP Olarak İndir
import shutil
from google.colab import files

SAVE_DIR = "yerelai_custom_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

shutil.make_archive("yerelai_custom_lora", "zip", SAVE_DIR)
files.download("yerelai_custom_lora.zip")
print("İndirilen ZIP içindeki dosyaları 'yerelai/models/loras/<adaptor_adi>/' klasörüne çıkartın.")

In [ ]:
# 6. (Önerilen) GGUF Turbo Hızı için LoRA Adaptörünü GGUF Formatına Dönüştür
# Bu dosya YerelAI Studio'da llama-server ile 90+ tok/sn GGUF Turbo hızında doğrudan çalışır!
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /content/llama_cpp_conv
!pip install -q gguf
!python /content/llama_cpp_conv/convert_lora_to_gguf.py \
    --base "{MODEL_NAME}" \
    --outfile yerelai_custom_lora_gguf.gguf \
    --outtype f16 \
    "{SAVE_DIR}"

from google.colab import files
files.download("yerelai_custom_lora_gguf.gguf")
print("Bu .gguf dosyasını 'yerelai/models/loras/<adaptor_adi>/' klasörüne koyun; YerelAI otomatik GGUF Turbo ile yükler!")